In [ ]:
def get_forward_backward_func():
    """Retrieves the appropriate forward_backward function given the
    configuration of parallel_state.

    Returns a function that will perform all of the forward and
    backward passes of the model given the pipeline model parallel
    world size and virtual pipeline model parallel world size in the
    global parallel_state.

    Note that if using sequence parallelism, the sequence length component of
    the tensor shape is updated to original_sequence_length /
    tensor_model_parallel_world_size.

    The function returned takes the following arguments:

    forward_step_func (required): A function that takes a data
        iterator and a model as its arguments and return the model's
        forward output and the loss function. The loss function should
        take one torch.Tensor and return a torch.Tensor of loss and a
        dictionary of string -> torch.Tensor.

        A third argument, checkpoint_activations_microbatch, indicates
        that the activations for this microbatch should be
        checkpointed. A None value for this argument indicates that
        the default from the configuration should be used. This is
        used when the
        num_microbatches_with_partial_activation_checkpoints is used.

        For example:

        def loss_func(loss_mask, output_tensor):
            losses = output_tensor.float()
            loss_mask = loss_mask.view(-1).float()
            loss = torch.sum(losses.view(-1) * loss_mask) / loss_mask.sum()

            # Reduce loss for logging.
            averaged_loss = average_losses_across_data_parallel_group([loss])

            return loss, {'lm loss': averaged_loss[0]}

        def forward_step(data_iterator, model):
            data, loss_mask = next(data_iterator)
            output = model(data)
            return output, partial(loss_func, loss_mask)


        forward_backward_func(forward_step_func=forward_step, ...)


    data_iterator (required): an iterator over the data, will be
        passed as is to forward_step_func. Expected to be a list of
        iterators in the case of interleaved pipeline parallelism.

    model (required): the actual model. Expected to be a list of modules in the case of interleaved
        pipeline parallelism. Must be a (potentially wrapped) megatron.core.models.MegatronModule.

    num_microbatches (int, required):
        The number of microbatches to go through

    seq_length (int, required): Sequence length of the current global batch. If this is a dual-stack
        transformer, this is the encoder's sequence length. This is ignored if variable_seq_lengths
        in the config is True. Otherwise, each microbatch in the current global batch size must use
        this sequence length.

    micro_batch_size (int, required): The number of sequences in a microbatch.

    decoder_seq_length (int, optional): The sequence length for the decoder in a dual-stack
        transformer. This is ignored for a single-stack transformer.

    forward_only (optional, default = False): Perform only the forward step

    collect_non_loss_data (optional, bool, default=False): TODO

    first_val_step (bool, optional): Is the first step of the validation phase. Used by
        Transformer Engine modules to only update their fp8 weights only on the first validation
        step.

    adjust_tensor_shapes_fn (Callable, optional): A function that adjusts the receive and send
        tensor shapes. Only applicable in forward_backward_pipelining_without_interleaving for now.
        Takes in a list of receive shapes and a list of send shapes and returns the adjusted
        respective list of shapes. Thus it is not used in the other forward-backward functions
        which have different shape handling.
        

    """
    pipeline_model_parallel_size = parallel_state.get_pipeline_model_parallel_world_size()
    
    if pipeline_model_parallel_size > 1:
        '''
        情况2: 有 pipeline 并行:
        
        '''
        # 子情况2.1: 使用虚拟 pipeline (Interleaved 模式)
        if parallel_state.get_virtual_pipeline_model_parallel_world_size() is not None:
            forward_backward_func = forward_backward_pipelining_with_interleaving
        
        # 子情况2.2: 传统 pipeline (无虚拟 pipeline)
        else:
            forward_backward_func = forward_backward_pipelining_without_interleaving
    else:
        '''
        无 pipeline 并行 (单设备/仅张量并行)
        '''
        forward_backward_func = forward_backward_no_pipelining
        
    return forward_backward_func

In [ ]:
pipeline_model_parallel_size?
│
├── 1 → forward_backward_no_pipelining
│
└── >1 → virtual_pipeline_model_parallel_world_size?
     │
     ├── 存在 → forward_backward_pipelining_with_interleaving (Interleaved)
     │
     └── 不存在 → forward_backward_pipelining_without_interleaving (传统 1F1B)

## 关键决策依据

| 决策条件                                      | 对应调度器                                      | 适用场景                                                                 |
|-----------------------------------------------|------------------------------------------------|--------------------------------------------------------------------------|
| `pipeline_model_parallel_size == 1`           | `forward_backward_no_pipelining`               | 单设备训练 / 仅使用张量并行（Tensor Parallelism）                        |
| `pipeline_model_parallel_size > 1` 且 无虚拟 pipeline | `forward_backward_pipelining_without_interleaving` | 传统 pipeline 并行（<font color='red'>每个设备只负责 1 个 stage</font> ）                        |
| `pipeline_model_parallel_size > 1` 且 <font color='red'>有虚拟 pipeline </font>| `forward_backward_pipelining_with_interleaving`  | Interleaved pipeline 并行（<font color='red'>单设备负责多个 stage，减少 pipeline 气泡</font>） |

虚拟 pipeline (Interleaved)：允许单个物理设备在 pipeline 中承担 多个逻辑 stage（例如 4-stage pipeline 中，2 个设备各负责 stage1+3 和 stage2+4），显著提升设备利用率。

方法在megatron/core/pilelin_parallel/schedules.py https://github.com/NVIDIA/Megatron-LM/blob/main/megatron/core/pipeline_parallel/schedules.py

- forward_backward_pipelining_with_interleaving https://github.com/NVIDIA/Megatron-LM/blob/main/megatron/core/pipeline_parallel/schedules.py#L889
- forward_backward_pipelining_without_interleaving https://github.com/NVIDIA/Megatron-LM/blob/main/megatron/core/pipeline_parallel/schedules.py#L2026
- forward_backward_no_pipelining https://github.com/NVIDIA/Megatron-LM/blob/main/megatron/core/pipeline_parallel/schedules.py#L591



## 整体功能说明：
1. 根据 parallel_state（并行状态）的配置，检索并返回合适的 forward_backward（前向-反向传播）函数。
    
2. <font color='red'>该函数会返回一个具体的执行函数，这个函数能够根据全局并行状态中的流水线模型并行大小（pipeline model parallel world size）
    和虚拟流水线模型并行大小（virtual pipeline model parallel world size），来执行模型的所有前向传播和反向传播过程。</font>
    
3. 注意：如果使用了序列并行（sequence parallelism），张量形状中的序列长度部分会被更新为 原始序列长度 / 张量模型并行大小。

## 🛠️ 返回函数的参数详解
原文翻译：
返回的这个函数需要接收以下参数：

1. forward_step_func（必填）
一个函数，接收数据迭代器（data iterator）和模型（model）作为参数，并返回模型的前向输出以及损失函数（loss function）。
这个损失函数应该接收一个 torch.Tensor，并返回一个标量损失值以及一个包含字符串到张量映射的字典。

第三个参数 checkpoint_activations_microbatch 表示该微批次（microbatch）的激活值是否需要进行重计算（activation checkpointing）。如果该参数为 None，则使用配置文件中的默认值。这通常在使用 num_microbatches_with_partial_activation_checkpoints 时会用到。

💡 解释：
这是你（用户）最需要自定义的部分。你需要告诉框架：“怎么跑一步前向传播”以及“怎么算 Loss”。
- 这里的 partial(loss_func, loss_mask) 就是典型的闭包用法，把当前批次特有的 loss_mask 提前打包好，等框架算完前向传播拿到 output_tensor 后，直接喂给这个损失函数。

In [ ]:
def loss_func(loss_mask, output_tensor):
    losses = output_tensor.float()
    loss_mask = loss_mask.view(-1).float()
    loss = torch.sum(losses.view(-1) * loss_mask) / loss_mask.sum()
    # 跨数据并行组平均损失以便记录日志
    averaged_loss = average_losses_across_data_parallel_group([loss])
    return loss, {'lm loss': averaged_loss[0]}

def forward_step(data_iterator, model):
    data, loss_mask = next(data_iterator)
    output = model(data)
    # 注意这里使用了 partial 来提前绑定 loss_mask
    return output, partial(loss_func, loss_mask)

2. data_iterator（必填）
- 数据的迭代器，会原封不动地传给 forward_step_func。如果是交错式流水线并行（interleaved pipeline parallelism），这里预期是一个迭代器的列表。
3. model（必填）
- 实际的模型。在交错式流水线并行中，预期是一个模块（module）的列表。它必须是一个（可能被包装过的）megatron.core.models.MegatronModule。
4. num_microbatches（必填，整数）
- 需要处理的微批次（microbatches）的总数量。
5. seq_length（必填，整数）
- 当前全局批次的序列长度。如果是双栈（dual-stack，比如编码器-解码器架构）Transformer，这指的是编码器的序列长度。如果配置中 variable_seq_lengths（可变序列长度）为 True，则忽略此参数；否则，当前全局批次中的每个微批次都必须使用这个序列长度。
6. micro_batch_size（必填，整数）
- 一个微批次中包含的序列（样本）数量。
7. decoder_seq_length（选填，整数）
- 双栈 Transformer 中解码器的序列长度。对于单栈 Transformer 会被忽略。
8. forward_only（选填，默认 False）
- 如果为 True，则只执行前向传播步骤（通常用于推理或验证阶段）。
9. collect_non_loss_data（选填，布尔值，默认 False）
- （TODO：待补充文档，通常用于收集除了 Loss 以外的其他指标数据）。
10. first_val_step（选填，布尔值）
- 表示这是否是验证阶段的第一个步骤。Transformer Engine 模块会用到它，以便仅在验证的第一步更新其 FP8（8位浮点数）权重统计量。
11. adjust_tensor_shapes_fn（选填，可调用函数）
- 一个用于调整接收和发送张量形状的函数。目前仅适用于非交错式的流水线并行（forward_backward_pipelining_without_interleaving）。它接收接收形状列表和发送形状列表，并返回调整后的相应形状列表。其他的前向-反向函数因为有各自的形状处理逻辑，所以不使用它。

## 总结一下：
- 这段注释其实就是在定义“分布式训练的标准化接口”。它告诉开发者：只要你按照这个规矩把数据迭代器、模型、微批次大小等参数传进来，并且写好 forward_step_func，框架就能自动帮你搞定背后极其复杂的流水线并行、梯度累积和通信逻辑。